# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

1000 images processed
2000 images processed
3000 images processed
4000 images processed
5000 images processed
6000 images processed
7000 images processed
8000 images processed
9000 images processed
10000 images processed
11000 images processed
12000 images processed


In [2]:
# Save test data
torch.save(test_loader, "test_data_regressor.pt")

In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [4]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.regressor = torch.nn.Sequential(torch.nn.Linear(2049, 256),
                                             torch.nn.Linear(256, 1))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.regressor(f)
        return y

In [5]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [6]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 525057


In [8]:
# Hyperparameters
epsilon = 1E-4

# Use MSE loss
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device,
            num_epochs=200, checkpoint_path='resnet_transfer_regressor_double_FULL.pth', graph_save_directory='./fully_trained_regressor_double')

Epoch [0/200], Train Loss: 23.58167, Val Loss: 11.48750 *, Train Acc: 0.08274, Validation Acc: 0.10722
Epoch [1/200], Train Loss: 14.24149, Val Loss: 10.38381 *, Train Acc: 0.12488, Validation Acc: 0.12722
Epoch [2/200], Train Loss: 11.72376, Val Loss: 10.42580, Train Acc: 0.15798, Validation Acc: 0.12500
Epoch [3/200], Train Loss: 11.40162, Val Loss: 10.37753 *, Train Acc: 0.14952, Validation Acc: 0.12611
Epoch [4/200], Train Loss: 11.05586, Val Loss: 10.62350, Train Acc: 0.15048, Validation Acc: 0.12722
Epoch [5/200], Train Loss: 10.92442, Val Loss: 10.87800, Train Acc: 0.14536, Validation Acc: 0.12333
Epoch [6/200], Train Loss: 10.34218, Val Loss: 10.42471, Train Acc: 0.15214, Validation Acc: 0.13222
Epoch [7/200], Train Loss: 10.66633, Val Loss: 10.63536, Train Acc: 0.15357, Validation Acc: 0.12611
Epoch [8/200], Train Loss: 10.04274, Val Loss: 10.12604 *, Train Acc: 0.14381, Validation Acc: 0.12389
Epoch [9/200], Train Loss: 9.96942, Val Loss: 10.00942 *, Train Acc: 0.16095, Valid

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [9]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_regressor_double_FULL.pth', criterion, test_loader, device)

Loss: 8.240287041134305
Accuracy: 0.13944444444444445
Off-by-One Accuracy: 0.3927777777777778


In [11]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_regressor_double_FULL.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-5},
    {'params': finetune_model.regressor.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device,
            num_epochs=200, checkpoint_path='resnet_finetuned_regressor_double_FULL.pth', graph_save_directory='./fully_trained_finetuned_regressor_double')

Epoch [0/200], Train Loss: 5.35202, Val Loss: 7.32499 *, Train Acc: 0.21952, Validation Acc: 0.16000
Epoch [1/200], Train Loss: 3.35260, Val Loss: 7.17056 *, Train Acc: 0.26333, Validation Acc: 0.15667
Epoch [2/200], Train Loss: 2.47351, Val Loss: 7.12896 *, Train Acc: 0.31524, Validation Acc: 0.16611
Epoch [3/200], Train Loss: 2.54857, Val Loss: 6.60775 *, Train Acc: 0.27940, Validation Acc: 0.16500
Epoch [4/200], Train Loss: 1.90264, Val Loss: 7.29912, Train Acc: 0.34905, Validation Acc: 0.15000
Epoch [5/200], Train Loss: 1.82115, Val Loss: 8.17651, Train Acc: 0.35869, Validation Acc: 0.14889
Epoch [6/200], Train Loss: 1.95542, Val Loss: 6.67739, Train Acc: 0.34250, Validation Acc: 0.16056
Epoch [7/200], Train Loss: 2.15624, Val Loss: 6.51754 *, Train Acc: 0.28774, Validation Acc: 0.15556
Epoch [8/200], Train Loss: 1.56883, Val Loss: 6.32788 *, Train Acc: 0.38452, Validation Acc: 0.16333
Epoch [9/200], Train Loss: 1.55894, Val Loss: 6.72850, Train Acc: 0.36845, Validation Acc: 0.1566

Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connect',
  address: '/run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock'
}
Unable to connect to VS Code server: Error in request.
Error: connect ENOENT /run/user/263712928/vscode-ipc-448c6089-e7cc-48ed-b103-7565292f72b2.sock
    at PipeConnectWrap.afterConnect [as oncomplete] (node:net:1278:16) {
  errno: -2,
  code: 'ENOENT',
  syscall: 'connec

In [12]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_finetuned_regressor_double_FULL.pth', criterion, test_loader, device)

Loss: 4.934102990892199
Accuracy: 0.14777777777777779
Off-by-One Accuracy: 0.5183333333333333
